# 01 - Estimate backbone: fixed-q vs AQF (single scenario)

Walks through the three estimator variants on one illustrative scenario: fixed-quantile baseline, oracle-AQF, and estimated-AQF with the identifiability check and soft-blend fallback.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import config
import estimators
import synth

In [ ]:
p, kappa = 0.3, 3.0
days_df = synth.generate_days(p=p, kappa=kappa, sigma=config.SIGMA, backbone_b=config.BACKBONE_B, days=config.DAYS, seed=1)
load = days_df["l"].to_numpy()

In [ ]:
# Fixed-quantile baseline
for q in config.FIXED_QS:
    print(q, estimators.fixed_quantile_backbone(load, q))

In [ ]:
# Oracle-AQF (true p, kappa known)
q_star_oracle = estimators.aqf_quantile(p, kappa)
print("q_star_oracle =", q_star_oracle)
print("B_hat_oracle =", estimators.fixed_quantile_backbone(load, q_star_oracle))

In [ ]:
# Estimated-AQF: fit mixture, check identifiability, blend toward the fallback default
fit = estimators.fit_mixture(load, seed=2)
d_hat = estimators.identifiability_diagnostic(fit.kappa_hat)
q_star_hat = estimators.aqf_quantile(fit.p_hat, fit.kappa_hat)
q_final, weight = estimators.fallback_blend(q_star_hat, config.Q_DEFAULT, d_hat, config.D_THRESH)
print(fit)
print("d_hat =", d_hat, " q_star_hat =", q_star_hat, " q_final =", q_final, " weight =", weight)